# SEC Filings Pipeline

Full pipeline from scraping 10-K filings through BigQuery AI extraction and property graph creation.

## Pipeline Steps
1. **Google Cloud authentication** – gcloud auth
2. **Scraper** – Download 10-K filings from SEC.gov
3. **Parser** – Parse SGML to Markdown (optional, for viewing)
4. **Extract Sections** – Parse SGML to JSONL (Item 1, 1A, 7; feeds BigQuery)
5. **Upload JSONL to GCS** – Sync JSON to BigQuery load bucket
6. **Init Tables** – Create sections, insights, staging tables
7. **Extraction** – AI extraction (Gemini) → insights
8. **Create Graph** – Transform insights to graph_edges
9. **Prepare Property Graph** – Node and edge tables
10. **Add Action to Market** – market_action column
11. **Create Property Graph DDL** – SecGraph property graph

## Configuration

Set the environment constants below. These variables control GCP authentication, local and cloud storage, and pipeline depth.

### GCP Settings
- **`GCP_PROJECT`**: Your Google Cloud Project ID. Find this in the [GCP Console Project Dashboard](https://console.cloud.google.com/home/dashboard). This project must have the BigQuery and Cloud Storage APIs enabled.
- **`GCS_BUCKET`**: The Google Cloud Storage bucket used for staging data for BigQuery load. Format: `gs://your-bucket-name`.
- **`BQ_LOCATION`**: The regional location for your BigQuery datasets (e.g., `US`, `EU`).

### Local Directories
These variables define where data is stored on your local machine or external drive.
- **`SGML_DIR`**: Root directory for raw `.sgml` files from SEC.gov.
- **`MARKDOWN_DIR`**: Directory for converted `.md` files.
- **`JSON_DIR`**: Directory for sectioned JSONL files (Item 1, 1A, 7).

### Scraper Settings
- **`SCRAPER_LIMIT`**: The number of companies to pull from the companies list (e.g., 5, 20, 500). Use a small number to test.
- **`SCRAPER_LAST_N_YEARS`**: The number of historical years to scrape for each company.

In [ ]:
# Configure these for your environment
import os

GCP_PROJECT = "kineviz-bigquery-graph"  # Change to your project ID
GCS_BUCKET = "gs://kineviz-fortune500-data"  # Change to your bucket
BQ_LOCATION = "US"

# Local data directories
SGML_DIR = "/Volumes/Dienert/Fortune500/data/sgml"
MARKDOWN_DIR = "/Volumes/Dienert/Fortune500/data/markdown"
JSON_DIR = "/Volumes/Dienert/Fortune500/data/json"

# Pipeline depth
SCRAPER_LIMIT = 500           # Companies from list.csv (use small number for testing)
SCRAPER_LAST_N_YEARS = 5      # Years of filings to fetch
SCRAPER_OUTPUT = SGML_DIR

# Derived project constants
BQ_PROJECT = GCP_PROJECT

## 0. Colab Setup
If you are running this in Google Colab, you need to clone the repository and install requirements to access the custom scraper scripts and SQL files.

In [10]:
import sys
import os
if 'google.colab' in sys.modules:
    from IPython import get_ipython
    ipy = get_ipython()
    if not os.path.exists('00.0_scraper.py'):
        ipy.system('git clone https://github.com/Kineviz/fortune500.git')
        os.chdir('fortune500')
ipy.system('pip install -r requirements.txt')


  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached thefuzz-0.22.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached markdownify-1.2.2-py3-none-any.whl.metadata (9.9 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached thefuzz-0.22.1-py3-none-any.whl (8.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.2 MB/s  0:00:00
Using cached markdownify-1.2.2-py3-none-any.whl (15 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [markdownify] [rapidfuzz]


## 1. Setup & Google Cloud Authentication

In [4]:
    # Using GCP_PROJECT from configuration above

if GCP_PROJECT != "YOUR_PROJECT_ID":
    from IPython import get_ipython
    ipy = get_ipython()
    ipy.system(f"gcloud config set project {GCP_PROJECT}")
    ipy.system("gcloud auth login")
    result = __import__("subprocess").run(
        ["gcloud", "config", "get-value", "project"],
        capture_output=True, text=True
    )
    current = (result.stdout or "").strip() or "(unset)"
    print(f"✓ Project verified: {current}") if current == GCP_PROJECT else print(f"⚠ Current: {current}")
else:
    raise ValueError("Set GCP_PROJECT in the Configuration cell at the beginning of the notebook.")


To update your Application Default Credentials quota project, use the `gcloud auth application-default set-quota-project` command.
Updated property [core/project].


Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=6kr8kaXYEzD3WiQG4FluTjGDOqxtB6&access_type=offline&code_challenge=qFF2eeCLcpND7k4MnsdYoiB25BwUEDEJZdOJfGcsiUg&code_challenge_method=S256


You are now logg

### Ensure we're in the project root (where 00.0_scraper.py, SQL files, list.csv live)


In [5]:
import os

def find_project_root():
    cwd = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(cwd, "00.0_scraper.py")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            break
        cwd = parent
    return os.getcwd()

root = find_project_root()
os.chdir(root)
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/dienert/projects/fortune500


### Verify GCS bucket exists and is accessible


In [8]:
import subprocess

result = subprocess.run(
    ["gsutil", "ls", GCS_BUCKET],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print(f"✓ Bucket accessible: {GCS_BUCKET}")
else:
    print(f"Bucket not found. Creating {GCS_BUCKET}...")
    create = subprocess.run(["gsutil", "mb", "-l", BQ_LOCATION, GCS_BUCKET], capture_output=True, text=True)
    if create.returncode == 0:
        print(f"✓ Bucket created: {GCS_BUCKET}")
    else:
        print(f"⚠ Failed to create bucket: {create.stderr or create.stdout}")

✓ Bucket accessible: gs://kineviz-fortune500-data


## 2. Run Scraper (00.0_scraper.py)

In [11]:
import asyncio
import sys

# Add project root to path
sys.path.insert(0, os.getcwd())

from importlib.util import spec_from_file_location, module_from_spec

# Load scraper module
spec = spec_from_file_location("scraper", "00.0_scraper.py")
scraper_mod = module_from_spec(spec)
spec.loader.exec_module(scraper_mod)

scraper = scraper_mod.SECScraper(
    limit=SCRAPER_LIMIT,
    last_n_years=SCRAPER_LAST_N_YEARS,
    workers=5,
    output_dir=SCRAPER_OUTPUT,
    dry_run=False,
)

# Scraper has internal tqdm; get expected count for context
import pandas as pd
try:
    n_companies = min(SCRAPER_LIMIT, len(pd.read_csv("list.csv")))
    print(f"Scraping up to {n_companies} companies ({SCRAPER_LAST_N_YEARS} years each)...")
except FileNotFoundError:
    n_companies = SCRAPER_LIMIT
await scraper.run()  # Jupyter has its own event loop; use await instead of asyncio.run()

Scraping up to 500 companies (5 years each)...
Fetching SEC tickers...
Loaded 10431 tickers.
Starting crawl for 500 unit(s) with 5 workers...


Scraping:  22%|██▏       | 112/500 [02:09<10:00,  1.55s/it]

[MOH] Failed to fetch results for 10-K: 503


Scraping:  24%|██▍       | 121/500 [02:24<11:19,  1.79s/it]

[DOW] Error processing row: HTTPSConnectionPool(host='www.sec.gov', port=443): Read timed out. (read timeout=30)


...

[URI] Failed to fetch results for 10-K: 503


Scraping:  61%|██████    | 303/500 [06:29<03:28,  1.06s/it]

[EME] Failed to fetch results for 10-K: 503


Scraping: 100%|██████████| 500/500 [11:13<00:00,  1.35s/it]

Done.


## 3. Run Parser (00.1_parser.py) (Optional)

In [ ]:
import subprocess
import sys
from tqdm.auto import tqdm

# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Parsing SGML to Markdown", unit="filing") as pbar:
    subprocess.run(
        [sys.executable, "00.1_parser.py", "--input_base", SGML_DIR, "--output_base", MARKDOWN_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

## 4. Extract Sections (01_extract_sections.py)

In [12]:
import subprocess
import sys
from tqdm.auto import tqdm

# Count SGML filings to process
n_sgml = sum(1 for r, d, f in os.walk(SGML_DIR) if "full-submission.txt" in f)
with tqdm(total=max(1, n_sgml), desc="Extracting sections to JSONL", unit="filing") as pbar:
    subprocess.run(
        [sys.executable, "01_extract_sections.py", "--input_base", SGML_DIR, "--output_base", JSON_DIR],
        check=True,
        cwd=os.getcwd(),
    )
    pbar.update(max(1, n_sgml))

/Users/dienert/miniconda3/envs/visio/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Extracting sections to JSONL:   0%|          | 0/2065 [00:00<?, ?filing/s]

Scanning /Volumes/Dienert/Fortune500/data/sgml...
Found 2065 filings to process.


Extracting sections to JSONL: 100%|██████████| 2065/2065 [11:09<00:00,  3.08filing/s]

Done.


## 5. Upload JSONL to GCS

In [13]:
import os
import glob
from IPython import get_ipython
from tqdm.auto import tqdm
json_src = os.path.abspath(JSON_DIR)
os.makedirs(json_src, exist_ok=True)
json_files = glob.glob(os.path.join(json_src, "*", "*", "sections.jsonl"))
n_json = len(json_files)
if n_json == 0:
    print(f"⚠ {json_src} is empty. Run Steps 2–4 (scraper, parser, extract sections) first, then re-run this cell.")
else:
    with tqdm(total=n_json, desc="Uploading JSONL to GCS", unit="file") as pbar:
        get_ipython().system(f'gsutil -m rsync -r {json_src}/ {GCS_BUCKET}/json')
        pbar.update(n_json)
    print(f"✓ Synced {n_json} section files to {GCS_BUCKET}/json")

Uploading JSONL to GCS:   0%|          | 0/1990 [00:00<?, ?file/s]


both the source and destination. Your crcmod installation isn't using the
module's C extension, so checksumming will run very slowly. If this is your
first rsync since updating gsutil, this rsync can take significantly longer than
usual. For help installing the extension, please see "gsutil help crcmod".

Building synchronization state...
If you experience problems with multiprocessing on MacOS, they might be related to https://bugs.python.org/issue33725. You can disable multiprocessing by editing your .boto config or by adding the following flag to your command: `-o "GSUtil:parallel_process_count=1"`. Note that multithreading is still available even if you disable multiprocessing.

Starting synchronization...
If you experience problems with multiprocessing on MacOS, they might be related to https://bugs.python.org/issue33725. You can disable multiprocessing by editing your .boto config or by adding the following flag to your command: `-o "GSUtil:parallel_process_count=1"`. Note that 

Uploading JSONL to GCS: 100%|██████████| 1990/1990 [01:20<00:00, 24.80file/s]

✓ Synced 1990 section files to gs://kineviz-fortune500-data/json


## 6. BigQuery Pipeline

### Util Functions

In [21]:
import subprocess
import glob
import json
from tqdm.auto import tqdm

SCHEMA = "filing_id:STRING,company:STRING,company_name:STRING,cik:STRING,sic:STRING,irs_number:STRING,state_of_inc:STRING,org_name:STRING,sec_file_number:STRING,film_number:STRING,business_street_1:STRING,business_street_2:STRING,business_city:STRING,business_state:STRING,business_zip:STRING,business_phone:STRING,mail_street_1:STRING,mail_street_2:STRING,mail_city:STRING,mail_state:STRING,mail_zip:STRING,filing_url:STRING,year:INTEGER,section_id:STRING,content:STRING"


def get_bq_distinct_companies() -> set[str]:
    sql = "SELECT DISTINCT company FROM `sec_filings.sections` WHERE company IS NOT NULL"
    cmd = [
        "bq",
        "query",
        "--use_legacy_sql=false",
        f"--location={BQ_LOCATION}",
        "--format=json",
        "--quiet",
        sql,
    ]
    bq_proj = globals().get("BQ_PROJECT", "")
    if bq_proj and bq_proj != "YOUR_PROJECT_ID":
        cmd.insert(-1, f"--project_id={bq_proj}")

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return set()

    try:
        rows = json.loads(result.stdout or "[]")
    except json.JSONDecodeError:
        return set()

    done: set[str] = set()
    for r in rows:
        v = r.get("company") if isinstance(r, dict) else None
        if isinstance(v, str) and v.strip():
            done.add(v.strip())
    return done


def run_bq_query(sql_file: str):
    sql_path = os.path.join(os.getcwd(), sql_file)
    if not os.path.exists(sql_path):
        for d in [os.getcwd()] + [os.path.dirname(os.getcwd())] * 3:
            p = os.path.join(d, sql_file)
            if os.path.exists(p):
                sql_path = p
                break
    with open(sql_path) as f:
        sql = f.read()
    cmd = ["bq", "query", "--use_legacy_sql=false", f"--location={BQ_LOCATION}"]
    proj = globals().get("BQ_PROJECT", "YOUR_PROJECT_ID")
    if proj and proj != "YOUR_PROJECT_ID":
        cmd.extend([f"--project_id={proj}"])
    subprocess.run(cmd, input=sql.encode(), check=True)


def run_bq_load(uris: str):
    cmd = ["bq", "load", "--source_format=NEWLINE_DELIMITED_JSON", "--replace", f"--schema={SCHEMA}", "sec_filings.sections_staging", uris]
    bq_proj = globals().get("BQ_PROJECT", "")
    if bq_proj and bq_proj != "YOUR_PROJECT_ID":
        cmd.insert(2, f"--project_id={bq_proj}")
    subprocess.run(cmd, check=True, capture_output=True)


def process_batch(uris: str):
    run_bq_load(uris)
    run_bq_query("04_extraction.sql")
    insert_cmd = ["bq", "query", "--use_legacy_sql=false", f"--location={BQ_LOCATION}",
         "INSERT INTO sec_filings.sections SELECT * FROM sec_filings.sections_staging"]
    bq_proj = globals().get("BQ_PROJECT", "")
    if bq_proj and bq_proj != "YOUR_PROJECT_ID":
        insert_cmd.insert(-1, f"--project_id={bq_proj}")
    subprocess.run(insert_cmd, check=True)

### 6.0 Initialize tables


In [15]:
sql_path = os.path.join(os.getcwd(), "03_init_tables.sql")
n_tables = 0
if os.path.exists(sql_path):
    with open(sql_path) as f:
        n_tables = sum(1 for line in f if "CREATE" in line and "TABLE" in line) or 1
with tqdm(total=max(1, n_tables), desc="Initializing BigQuery tables", unit="table") as pbar:
    run_bq_query("03_init_tables.sql")
    pbar.update(max(1, n_tables))
print("Tables initialized.")

Waiting on bqjob_r6bcb0408eb171cf6_0000019cf932924c_1 ... (4s) Current status: DONE   
Initializing BigQuery tables: 100%|██████████| 2/2 [00:09<00:00,  4.58s/table]

CREATE OR REPLACE TABLE sec_filings.sections
(
    filing_id STRING,
    company STRING,
    company_name STRING,
    cik STRING,
    sic STRING,
    irs_number STRING,
    state_of_inc STRING,
    org_name STRING,
    sec_file_number STRING,
    film_number STRING,
    business_street_1 STRING,
    business_street_2 STRING,
    business_city STRING,
    business_state STRING,
    business_zip STRING,
    business_phone STRING,
    mail_street_1 STRING,
    mail_street_2 STRING,
    mail_city STRING,
    mail_state STRING,
    mail_zip STRING,
    filing_url STRING,
    year INT64,
    section_id STRING,
    content STRING
); -- at [2:1]
Replaced kineviz-bigquery-graph.sec_filings.sections

CREATE OR REPLACE TABLE sec_filings.insights AS
SELECT * FROM 
  ML.GENERATE_TEXT(
    MODEL `sec_filings.gemini_pro`,
    (SELECT 'dummy' AS prompt, * FROM sec_filings.sections LIMIT 0), 
    STRUCT(0.2 AS temperature, 8192 AS max_output_tokens, FALSE AS flatten_json_output)
  )
LIMIT 0; -- at [32:

### 6.1 Batch load, extract, archive


In [27]:
import subprocess
from tqdm.auto import tqdm

START_OVER = False  # Set True to re-run everything (run 6.0 Init Tables first to avoid duplicates)

company_dirs_all = sorted(glob.glob(os.path.join(JSON_DIR, "*")))
company_dirs_all = [d for d in company_dirs_all if os.path.isdir(d)]

already_done = set() if START_OVER else get_bq_distinct_companies()

company_dirs = company_dirs_all
if already_done and not START_OVER:
    n_total = len(company_dirs_all)
    n_done = sum(1 for d in company_dirs_all if os.path.basename(d) in already_done)
    pct_done = (n_done / n_total * 100) if n_total else 0.0

    ans = input(
        f"Found {n_done}/{n_total} companies already in BigQuery ({pct_done:.1f}%). Resume from where it stopped? [Y/n] "
    ).strip().lower()
    if ans in ("n", "no"):
        print(
            "Not resuming. To start over, set START_OVER=True above (and run 6.0 Init Tables first to avoid duplicates)."
        )
        RUN_BATCH = False
    else:
        RUN_BATCH = True
        company_dirs = [d for d in company_dirs_all if os.path.basename(d) not in already_done]
        print(f"Resuming: processing {len(company_dirs)}/{n_total} remaining companies.")
else:
    RUN_BATCH = True
    if START_OVER:
        print(
            "START_OVER=True: processing all companies (run 6.0 Init Tables first to avoid duplicates)."
        )

all_sections = []
for d in company_dirs:
    all_sections.extend(glob.glob(os.path.join(d, "*", "sections.jsonl")))
n_section_files = len(all_sections)
n_companies = len(company_dirs)

gcs_prefix = f"{GCS_BUCKET}/json"

if RUN_BATCH:
    for idx, company_dir in enumerate(
        tqdm(
            company_dirs,
            desc=f"Processing {n_companies} companies ({n_section_files} section files)",
            unit="company",
        )
    ):
        sections = glob.glob(os.path.join(company_dir, "*", "sections.jsonl"))
        if not sections:
            continue
        uris = [
            f"{gcs_prefix}/" + os.path.relpath(p, JSON_DIR).replace(os.sep, "/")
            for p in sections
        ]
        uris_str = ",".join(uris)
        company_ticker = os.path.basename(company_dir)
        pct = (idx + 1) / n_companies * 100
        try:
            process_batch(uris_str)
        except subprocess.CalledProcessError as e:
            err_detail = (
                (e.stderr or e.stdout or b"").decode(errors="replace").strip()
                if (e.stderr or e.stdout)
                else str(e)
            )
            print(f"\n⚠ [{company_ticker}] FAILED at {pct:.1f}% ({idx + 1}/{n_companies}): {e}")
            if err_detail:
                print(f"   {err_detail[:500]}")
            continue
        except Exception as e:
            print(
                f"\n⚠ [{company_ticker}] FAILED at {pct:.1f}% ({idx + 1}/{n_companies}): {type(e).__name__}: {e}"
            )
            continue

KeyboardInterrupt: Interrupted by user

### 6.2 Create graph edges


In [23]:
with tqdm(total=1, desc="Creating graph edges", unit="step") as pbar:
    run_bq_query("05_create_graph.sql")
    pbar.update(1)
print("Graph edges created.")

Waiting on bqjob_rab4249d7c448e56_0000019cfc5e515c_1 ... (3s) Current status: DONE   
Creating graph edges: 100%|██████████| 1/1 [00:07<00:00,  7.99s/step]

Replaced kineviz-bigquery-graph.sec_filings.graph_edges

Graph edges created.


### 6.3 Prepare node/edge tables


In [24]:
with tqdm(total=1, desc="Preparing node/edge tables", unit="step") as pbar:
    run_bq_query("06_prepare_property_graph.sql")
    pbar.update(1)
print("Property graph tables created.")

Waiting on bqjob_rcc28769527bf460_0000019cfc5e7133_1 ... (50s) Current status: DONE   
Preparing node/edge tables: 100%|██████████| 1/1 [00:56<00:00, 56.07s/step]

CREATE OR REPLACE TABLE sec_filings.nodes_company AS
SELECT DISTINCT 
  source_node AS id, 
  company_name AS label,
  cik,
  sic,
  irs_number,
  state_of_inc,
  org_name,
  sec_file_number,
  film_number,
  business_street_1,
  business_street_2,
  business_city,
  business_state,
  business_zip,
  business_phone,
  mail_street_1,
  mail_street_2,
  mail_city,
  mail_state,
  mail_zip
FROM sec_filings.graph_edges
WHERE source_label = 'Company'; -- at [4:1]
Replaced kineviz-bigquery-graph.sec_filings.nodes_company

CREATE OR REPLACE TABLE sec_filings.nodes_market AS
SELECT 
  edge_id AS id, 
  target_node AS label, 
  year, 
  section_id AS section, 
  filing_url AS link,
  properties AS evidence 
FROM sec_filings.graph_edges
WHERE target_label = 'Market' AND target_node IS NOT NULL; -- at [30:1]
Replaced kineviz-bigquery-graph.sec_filings.nodes_market

CREATE OR REPLACE TABLE sec_filings.nodes_risk AS
SELECT 
  edge_id AS id, 
  target_node AS label, 
  year, 
  section_id AS section

### 6.4 Add market_action to nodes_market


In [25]:
# Note: If market_action column already exists, ALTER TABLE will fail on re-run.
# Edit 06_1_add_action_to_market.sql to remove the ADD COLUMN line in that case.
with tqdm(total=1, desc="Adding market_action to nodes_market", unit="step") as pbar:
    run_bq_query("06_1_add_action_to_market.sql")
    pbar.update(1)
print("Market action column added.")

Waiting on bqjob_r4bc885aa15a64742_0000019cfc5f4bf1_1 ... (4s) Current status: DONE   
Adding market_action to nodes_market: 100%|██████████| 1/1 [00:09<00:00,  9.13s/step]

ALTER TABLE sec_filings.nodes_market
ADD COLUMN market_action STRING; -- at [3:1]
Altered kineviz-bigquery-graph.sec_filings.nodes_market

UPDATE sec_filings.nodes_market AS m
SET market_action = 
  CASE
    WHEN EXISTS (
      SELECT 1 
      FROM sec_filings.edges_entering e 
      WHERE e.target_node = m.id
    ) THEN 'Entering'
    WHEN EXISTS (
      SELECT 1 
      FROM sec_filings.edges_exiting e 
      WHERE e.target_node = m.id
    ) THEN 'Exiting'
    WHEN EXISTS (
      SELECT 1 
      FROM sec_filings.edges_expanding e 
      WHERE e.target_node = m.id
    ) THEN 'Expanding'
    ELSE NULL
  END
WHERE TRUE; -- at [7:1]
Number of affected rows: 9871

Market action column added.


### 6.5 Create SecGraph property graph


In [26]:
with tqdm(total=1, desc="Creating SecGraph property graph", unit="step") as pbar:
    run_bq_query("07_create_property_graph_ddl.sql")
    pbar.update(1)
print("Pipeline complete.")

Waiting on bqjob_r61031179ac3f4abb_0000019cfc5f6e5d_1 ... (0s) Current status: DONE   
Creating SecGraph property graph: 100%|██████████| 1/1 [00:05<00:00,  5.05s/step]

Pipeline complete.
